# OpenAI Agents SDK basics

## Learning goals

- Understand what `Agent`, `Runner`, and `RunResult` each own
- Configure an agent without making a network call
- Run asynchronously in Jupyter with an explicit turn limit
- Choose a conversation-state strategy without duplicating history
- Inspect safe result surfaces instead of treating the SDK as a black box


## Install the SDK

The PyPI package is named `openai-agents`, while Python imports use `agents`:

```text
uv add openai-agents
```

This repository already declares it, so learners should run `uv sync`. The SDK uses the OpenAI API key from `OPENAI_API_KEY` for live OpenAI runs.


## The three core objects

- **`Agent`** is mostly configuration: name, instructions, model, tools, handoffs, guardrails, output type, and model settings. Constructing one does not call a model.
- **`Runner`** executes the SDK's agent loop. It calls the current agent, handles function tools and handoffs, and stops on a final output or a runtime condition.
- **`RunResult`** contains the final output plus rich metadata such as new run items, the last active agent, raw model responses, guardrail results, usage, and continuation helpers.

The SDK packages the loop we built from scratch; it does not remove the need for narrow tools, authorization, budgets, validation, and safe logging.


## Mental model

```mermaid
flowchart LR
  A[User input] --> B[Runner.run]
  C[Agent configuration] --> B
  D[Application context and limits] --> B
  B --> E[Current agent model call]
  E --> F{Model output}
  F -->|final answer| G[RunResult]
  F -->|function tool| H[Validate and execute tool]
  F -->|handoff| I[Change active agent]
  H --> E
  I --> E
  G --> J[final_output / new_items / last_agent / usage]
```

One **runner turn** is one model invocation for the current agent. A single run can contain several turns because tools and handoffs feed observations back into the loop. `max_turns` is therefore a model-call limit, not a user-chat-turn limit.


## Load `.env` from the project root

Notebook working directories vary. Find `pyproject.toml` rather than assuming the kernel starts in the repository root. Existing shell variables keep priority through `override=False`.


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find pyproject.toml.")


project_root = find_project_root()
dotenv_loaded = load_dotenv(project_root / ".env", override=False)
print("Project root:", project_root)
print(".env found:", dotenv_loaded)
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))


## Construct an agent offline

Keep the model configurable. `ModelSettings` applies request-level behavior for this agent; here we cap generated tokens and explicitly disable response storage for the stateless classroom example. Availability and supported settings still depend on the selected model.


In [ ]:
from agents import Agent, ModelSettings, Runner

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

travel_agent = Agent(
    name="Travel helper",
    handoff_description="Handles practical travel-planning questions.",
    instructions=(
        "Help with travel planning. State assumptions, distinguish estimates "
        "from confirmed facts, and do not claim to complete bookings."
    ),
    model=OPENAI_MODEL,
    model_settings=ModelSettings(max_tokens=300, store=False),
)

print("Agent:", travel_agent.name)
print("Model:", travel_agent.model)
print("Tools:", len(travel_agent.tools))
print("Handoffs:", len(travel_agent.handoffs))


## Choose the correct Runner mode

| Method | Behavior | Typical use |
|---|---|---|
| `await Runner.run(...)` | Asynchronous; returns `RunResult` | Jupyter, async web servers, concurrent apps |
| `Runner.run_sync(...)` | Synchronous wrapper | Simple scripts without a running event loop |
| `Runner.run_streamed(...)` | Asynchronous streaming result | Token/event streaming and responsive UIs |

Jupyter already runs an event loop, so use top-level `await Runner.run(...)`. Do not call `asyncio.run(...)` inside a normal notebook cell. Always set a finite `max_turns` unless you have a carefully justified runtime policy.


## Optional live run

This call is disabled by default because it uses the network and may incur cost. When enabled, the final output is the simplest user-facing result surface.


In [ ]:
RUN_LIVE_CALL = False

if RUN_LIVE_CALL:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY before enabling the live run.")

    result = await Runner.run(
        travel_agent,
        "Plan a relaxed two-day visit to Jaipur.",
        max_turns=5,
    )
    print(result.final_output)
    print("Last agent:", result.last_agent.name)
    print("New run items:", len(result.new_items))
else:
    print("Live Agents SDK run skipped.")


## Read the result at the right level

| Need | Result surface |
|---|---|
| Final answer | `result.final_output` |
| Agent that finished | `result.last_agent` |
| Rich tool/handoff/message events | `result.new_items` |
| Replay-ready local input | `result.to_input_list()` |
| Latest Responses continuation ID | `result.last_response_id` |
| Provider-level diagnostics | `result.raw_responses` |
| Guardrail outcomes | guardrail result arrays |
| Usage totals | `result.context_wrapper.usage` |

`final_output` is a string unless the final agent defines `output_type`. With handoffs, the finishing agent can change, so application code should know which output contracts are possible. Do not display raw responses or `new_items` directly in a public UI; extract an allowlisted view.


## Choose one conversation-state strategy

The SDK supports several continuation patterns:

1. `result.to_input_list()` for explicit application-managed history;
2. `session=...` for SDK-managed persistence through a session implementation;
3. `previous_response_id` for lightweight OpenAI Responses continuation;
4. `conversation_id` for an OpenAI-managed conversation resource.

Usually choose one per conversation. Combining full client-managed history with server-managed continuation can duplicate context.


In [ ]:
from agents import SQLiteSession

# In-memory teaching session; a file path would persist SQLite data.
teaching_session = SQLiteSession("travel-demo", db_path=":memory:")
print(type(teaching_session).__name__)


## Failure cases and improvements

| Failure | Improvement |
|---|---|
| `ModuleNotFoundError: agents` | Run `uv sync` and select the course kernel |
| Authentication error | Verify `OPENAI_API_KEY` without printing it |
| Agent loops until `MaxTurnsExceeded` | Reduce tool ambiguity and enforce a finite `max_turns` |
| Notebook event-loop error | Use `await Runner.run`, not `Runner.run_sync`/`asyncio.run` |
| Follow-up repeats the full history | Pick one state strategy and avoid duplicate replay |
| UI leaks tool arguments or raw responses | Map result items to a redacted, allowlisted event view |


## Exercise

1. Clone `travel_agent` with stricter instructions using `travel_agent.clone(...)`.
2. Explain how `max_turns` differs from chat turns and tool-call count.
3. Compare manual `to_input_list()` continuation with `SQLiteSession`.
4. Enable one live run and inspect only `final_output`, `last_agent.name`, and item type names.
5. Map each SDK primitive to the hand-written runtime from notebook `08_agent_loop_from_scratch`.


## Official references

- [OpenAI Agents SDK guide](https://developers.openai.com/api/docs/guides/agents)
- [Agents SDK quickstart](https://openai.github.io/openai-agents-python/quickstart/)
- [Running agents](https://openai.github.io/openai-agents-python/running_agents/)
- [Run results](https://openai.github.io/openai-agents-python/results/)


## Summary

`Agent` defines capabilities and behavior, `Runner` executes the bounded model/tool/handoff loop, and `RunResult` exposes final output plus operational detail. In Jupyter, run asynchronously, keep live calls guarded, choose one history strategy, and inspect only the result surfaces your application actually needs.
